In [12]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import networkx as nx
import random
import heapq
import collections

### All the simulation uses hours and kilometers

In [13]:
LAMBDA_T = [314.2, 162.4, 138.6, 148.8, 273.2, 1118.8, 2773.8, 4036.2, 4237.4, 3277.0, 2843.0, 2876.4, 3143.0, 3277.8, 3546.2, 4335.0, 4945.4, 4525.8, 2847.8, 1828.0, 1378.4, 1271.2, 1171.2, 767.6]

In [98]:
#Sampling arrival times of cars to network
def lambdat(t : np.array):
    lambdat = []
    for time in t:
        lambdat.append(LAMBDA_T[int(np.floor(time))])
    return lambdat

def arrival_times(lam): #Taken from lecture notes
    max_T = 24
    arrival_times = collections.deque()
    exp_dist = stats.expon(scale = 1/lam)
    t = exp_dist.rvs()
    while t < max_T:
        arrival_times.append(t)
        t += exp_dist.rvs()
    
    return np.asarray(arrival_times)

In [99]:
Graph = nx.read_gml('networkAssignment.gml')
JUNCTIONS = list(Graph.nodes)
DIC_EDGES = {}
for edge in Graph.edges:
    DIC_EDGES[edge] = []
    #Associate departure from queue distribution to each edge based on number of lanes
    rate = Graph.edges[edge]['lanes']
    Graph.edges[edge]['DepDist'] = stats.expon(scale = 1/rate)

In [100]:
class FES:
    def __init__(self):
        self.events = []

    def add(self, event):
        heapq.heappush(self.events, event)
    
    def next(self):
        return heapq.heappop(self.events)
    
    def isEmpty(self):
        return len(self.events) == 0
    
    def __repr__(self):
        string = ''
        sorted_events = sorted(self.events)
        for event in sorted_events:
            string += f'{event}\n'
        return string

In [116]:
class Queue:
    def __init__(self, road):
        self.road = road
        self.cars = []

    def add(self, car, time):
        heapq.heappush(self.cars, car)
        car.enter_queue(time)

    def add_list(self, list, time):
        for car in list:
            heapq.heappush(self.cars, car)
            car.enter_queue(time)   

    def next(self):
        return heapq.heappop(self.cars)
    
    def first(self):
        return heapq.nsmallest(1, self.cars)
    
    def isEmpty(self):
        return len(self.car) == 0
    
    def __repr__(self):
        string = ''
        sorted_cars = sorted(self.cars)
        for car in sorted_cars:
            string += f'{car}\n'
        return string

In [102]:
class Event:
    TYPE = ['New car', 'Car departure', 'Accident', 'Accident end', 'left queue']
    def __init__(self, typ:int, time, car = None, road = None):
        #types:
            #0 : Arrival of car to the network
            #1 : Car leaves current road and goes on to the next
            #2 : Accident in road
        self.type = typ
        self.time = time
        self.road = road

        if typ == 0:
            car = Car(time_entrance = time)
    
        self.car = car
        
    def __str__(self):
        if self.type == 0:
            return f'{self.TYPE[self.type]} from {self.car.origin} to {self.car.destination} at {self.time}'
        if self.type == 1:
            return f'{self.TYPE[self.type]} of {self.car} at {self.time}h'
        if self.type == 2:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'
        if self.type == 3:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'
        if self.type == 4:
            return f'{self.car} {self.TYPE[self.type]} at {self.road} at {self.time}h'

    def __lt__(self, other):
        return self.time < other.time
    
    def new_time(self, new_time):
        self.time = new_time

In [ ]:
class Car:
    VELOCITIES = [100, 80]
    VELOCITIES_P = [0.9, 0.1]
    def __init__(self, time_entrance, origin = None, destination = None):
        #Origin and destination
        origin, destination = np.random.choice(JUNCTIONS, 2, replace = False)
        
        self.origin = origin
        self.destination = destination

        #path to follow
        self.path = nx.shortest_path(Graph, self.origin, self.destination, weight = 'length')

        #Velocity
        self.velocity = np.random.choice(self.VELOCITIES, p=self.VELOCITIES_P)

        #Variable to keep track how far into the path we are (to simplify scheduling events)
        #Int between 0 and len(path) - 1 that indicates in which edge we are, starting at 0
        #Essentially, how many edges has it travelled so far
        self.progress = 0
        
        #Time entrance
        self.time = time_entrance

        #Schedule next event and store it as attribute
        self.next_event = self.schedule_event_exit()


    def __str__(self):
        return f'Vehicle travelling from {self.origin} to {self.destination} at {self.velocity} km/h, atm at {self.path[self.progress]}'
    
    def schedule_event_exit(self):

        #Remove car from list of cars in previous edge
        if self.progress > 0:
            #Edges can be expressed in two directions, account for it
            edge = (self.path[self.progress - 1], self.path[self.progress])
            if edge in DIC_EDGES.keys():
                DIC_EDGES[edge].remove(self)
            else:
                DIC_EDGES[edge[::-1]].remove(self)


        if  self.progress < len(self.path) - 1: 
            #find next edge to travel through and its length
            edge = Graph.edges[(self.path[self.progress], self.path[self.progress + 1])]
            length = edge['length']

            #Sample travel time of edge
            mean = length / (self.velocity /3.6) #seconds
            std = mean / 20
            time_to_travel = np.random.normal(loc = mean, scale = std) / 3600 #back to hours

            new_time = self.time + time_to_travel

            #Store event and increase progress
            self.next_event = Event(1 , new_time, car=self)
            self.increase_progress()
            self.increase_time(new_time)

            #Add car to list of cars in the new edge
            edge = (self.path[self.progress - 1], self.path[self.progress])
            if edge in DIC_EDGES.keys():
                DIC_EDGES[edge].append(self)
            else:
                DIC_EDGES[edge[::-1]].append(self)

            return self.next_event
        
        # if self.progress == len(self.path) - 1:
        #     print('Car has reached its destination')
        #     self.travel_time = self.next_event.time
    
    def increase_progress(self):
        self.progress += 1

    def increase_time(self, new_time):
        self.time = new_time

    def __lt__(self, other):
        return self.time < other.time
    
    def enter_queue(self, time):
        #store time at which the car entered the queue
        self.time_enter = time

    def exit_queue(self, time):
        #Update time of car when it left the queue
        self.time += time - self.time_enter

        #Update time of event leaving road
        self.next_event.new_time(self.time)

In [104]:
#Using a thining approach
max_lambda = np.max(LAMBDA_T) + 1
all_arrivals = arrival_times(max_lambda)

uniform_dist = stats.uniform(0,1)
u_rvs = uniform_dist.rvs(len(all_arrivals))
accept_filter = u_rvs * max_lambda < lambdat(all_arrivals)

accepted_arrivals = all_arrivals[accept_filter]

In [105]:
DIC_EDGES.keys()

dict_keys([('1410566272', '8432860337'), ('1410566272', '44996729'), ('1410566272', '45098337'), ('43655900', '44112758'), ('43655900', '1558293339'), ('43655900', '672759992'), ('2752332143', '44708181'), ('2752332143', '1558293339'), ('2752332143', '44112758'), ('2752332143', '44153932'), ('43349094', '43108886'), ('43349094', '1558293339'), ('43349094', '44153932'), ('42995944', '43003075'), ('42995944', '672759992'), ('8432860337', '44153932'), ('8432860337', '44996729'), ('43108886', '43003075'), ('44708181', '44112758'), ('44708181', '45098337'), ('672759992', '42549870'), ('43003075', '1558293339')])

In [112]:
#Simulation (can be turned into an object later)
LIST_CARS = []
fes = FES()
for arrival in accepted_arrivals:
    #Two events associated with each arrival
    arrival_event = Event(0, arrival)
    fes.add(arrival_event)

In [113]:
fes.add(Event(2, 1, road=('43108886', '43003075')))
fes.add(Event(3, 1.5, road=('43108886', '43003075')))

In [117]:
#Queue for accidents setup
Queues = {}
for edge in DIC_EDGES.keys():
    Queues[edge] = Queue(edge)

In [ ]:
t = 0 #current time
while t < 24.0:
    event = fes.next()
    t = event.time

    #Car joins network
    if event.type == 0:
        car_travel_event = event.car.next_event
        fes.add(car_travel_event)
        LIST_CARS.append(event.car)

    #Car leaves road
    if event.type == 1:
        next_travel_event_car = event.car.schedule_event_exit()
        if type(next_travel_event_car) == Event: #If the car has arrived to its destination it wont return an event object
            fes.add(next_travel_event_car)
        # else:
            # print(event.car)

    #Start accident
    if event.type == 2:
        road = event.road
        #choose random cars affected:
        amount_cars_affected = np.random.randint(len(DIC_EDGES[road]))
        cars_affected = np.random.choice(DIC_EDGES[road], amount_cars_affected, replace=False)

        Queues[road].add_list(cars_affected, event.time)

        #Assuming end of accident event is already created, else create here

    #End accident
    if event.type == 3:
        #1 minute until first car leaves the queue 
        first_departure_event = Event(4, event.time + 1/60, road=event.road)
        fes.add(first_departure_event)

    #Departure from road
    if event.type == 4:
        road = event.road
        time = event.time
        #Remove car from queue
        car = Queues[road].next()

        #Modify time to arrival of next node in path
        car.exit_queue(time)

        #Next departure event
        next_event_time = time + Graph.edges[road]['DepDist'].rvs()
        
        next_event = Event(4, next_event_time, road=road)
        fes.add(next_event)

    

In [119]:
#Proof that updating the car event also updates the one in the list
car_debug = Car(0)
list_debug = [car_debug.next_event]
print(list_debug[0].time)
car_debug.enter_queue(1)
car_debug.exit_queue(2)
print(list_debug[0].time)

0.19555647293644068
1.1955564729364407


In [121]:
#Checking if cars make it to destionation
for i in range(0, len(LIST_CARS)):
    car_i = LIST_CARS[i]
    # if car_i.progress != len(car_i.path) - 1:
    #     print(f'oh oh {LIST_CARS[i].time}')
    if car_i.path[car_i.progress] != car_i.destination:
        print(f'oh oh {LIST_CARS[i].time}')

oh oh 24.02555020847363
oh oh 24.07303182781705
oh oh 24.020168019477133
oh oh 24.031199225908658
oh oh 24.004351569863175
oh oh 24.024846959698156
oh oh 24.028803948611916
oh oh 24.020239002763002
oh oh 24.096275702310916
oh oh 24.115349769969566
oh oh 24.06138928395156
oh oh 24.14112505048084
oh oh 24.131304714520446
oh oh 24.040935099129346
oh oh 24.164347339588275
oh oh 24.08506456138264
oh oh 24.07618271248735
oh oh 24.076507567102123
oh oh 24.020221910879485
oh oh 24.010587396882677
oh oh 24.046178723937587
oh oh 24.090140668922587
oh oh 24.014724919372657
oh oh 24.100716449261466
oh oh 24.00108367385353
oh oh 24.021005594122464
oh oh 24.00337781508666
oh oh 24.02960032958916
oh oh 24.056820229809173
oh oh 24.066055160752924
oh oh 24.087054974219676
oh oh 24.098807676852804
oh oh 24.02327882842002
oh oh 24.068810462845647
oh oh 24.07955958396111
oh oh 24.040310811351464
oh oh 24.138902799853728
oh oh 24.119580864611304
oh oh 24.0747145734824
oh oh 24.095494132455755
oh oh 24.0414